# Tutorial 7 — 3D Geometry & Camera Models

This tutorial covers the **3D geometry** classes from `le_geometry` and
visualizes them using [Rerun](https://rerun.io) directly in Jupyter.

**Classes covered:**

| Class | Description |
|-------|-------------|
| `Line3` / `_f64` | Infinite 3D line (point + direction) |
| `LineSegment3` / `_f64` | Bounded 3D line segment |
| `Plane` / `_f64` | Infinite plane (point + normal) |
| `Pose` / `_f64` | Position + orientation (6 DOF) |
| `Camera` / `_f64` | Intrinsic + extrinsic camera model |
| `CameraHom` / `_f64` | Homogeneous line projection |
| `CameraPluecker` / `_f64` | Pl\u00fccker line projection |
| `Camera2P` / `_f64` | Two-point line projection |
| `CameraCV` / `_f64` | OpenCV-compatible camera + line proj |

**Prerequisites:**
- Built bindings: `bazel build //libs/...`
- `pip install "rerun-sdk[notebook]"`

**Sections:**

1. Setup & Imports
2. Line3 — Infinite Lines in 3D
3. LineSegment3 — Bounded Segments
4. Plane — Infinite Planes
5. Line-Plane Intersections
6. Pose — Position & Orientation
7. Camera Models
8. Projection — 3D to 2D
9. Rerun 3D Visualization
10. Comprehension Questions
11. Challenge Exercise

---
## 1. Setup & Imports

In [ ]:
import sys, pathlib

workspace = pathlib.Path.cwd()
while not (workspace / "MODULE.bazel").exists():
    if workspace == workspace.parent:
        raise RuntimeError("Cannot find LineExtraction workspace root (MODULE.bazel)")
    workspace = workspace.parent

for lib in ["imgproc", "edge", "geometry", "eval", "lsd", "algorithm"]:
    p = workspace / f"bazel-bin/libs/{lib}/python"
    if p.exists():
        sys.path.insert(0, str(p))
    else:
        print(f"Warning: Not found: {p}  \u2014 run: bazel build //libs/{lib}/...")

sys.path.insert(0, str(workspace / "python"))

import numpy as np
import matplotlib.pyplot as plt
%matplotlib inline

import le_geometry as geo

# Rerun for 3D visualization
try:
    import rerun as rr
    HAS_RERUN = True
    print(f"Rerun {rr.__version__} available")
except ImportError:
    HAS_RERUN = False
    print("Rerun not installed \u2014 3D visualization cells will be skipped.")
    print("Install with: pip install 'rerun-sdk[notebook]'")

print(f"Workspace: {workspace}")
print(f"le_geometry loaded: {len([x for x in dir(geo) if not x.startswith('_')])} symbols")

---
## 2. Line3 — Infinite Lines in 3D

`Line3` represents an infinite line defined by a **point** and **direction**.

**Construction:**
- `Line3(px, py, pz, vx, vy, vz)` — point + direction
- `Line3.two_point(p1, p2)` — from two points

**Properties:** `origin`, `direction`, `momentum`

**Methods:** `distance_to_point()`, `distance_to_line()`, `nearest_point()`,
`angle_to()`, `translate()`, `rotate()`, `cayley()`, `from_cayley()`

In [ ]:
# --- Construct a Line3 ---
line_a = geo.Line3(0, 0, 0, 1, 0, 0)        # along x-axis
line_b = geo.Line3.two_point(0, 0, 0, 1, 1, 1)  # diagonal

print(f"Line A: origin={line_a.origin}, direction={line_a.direction}")
print(f"Line B: origin={line_b.origin}, direction={line_b.direction}")
print(f"Line B momentum: {line_b.momentum}")

In [ ]:
# --- Distance and angle ---
dist = line_a.distance_to_point(0.0, 3.0, 4.0)
nearest = line_a.nearest_point(0.0, 3.0, 4.0)
print(f"Distance from (0, 3, 4) to x-axis: {dist:.2f}")
print(f"Nearest point on x-axis: {nearest}")

# Angle between two lines
angle = line_a.angle_to(line_b)
print(f"Angle between x-axis and diagonal: {np.degrees(angle):.2f}°")

In [ ]:
# --- Transform lines ---
import math

translated = geo.Line3(0, 0, 0, 1, 0, 0)  # copy of line_a
translated.translate(0, 5, 0)
print(f"Translated: origin={translated.origin}")

# Rotate 90 degrees around z-axis (Rodrigues vector)
rotated = geo.Line3(0, 0, 0, 1, 0, 0)  # copy of line_a
rotated.rotate(0, 0, math.pi / 2)
print(f"Rotated 90° around Z: direction={rotated.direction}")

In [ ]:
# --- Cayley representation ---
cayley = line_a.cayley()
print(f"Cayley representation: {cayley}")

# Reconstruct from Cayley — unpack the tuple
reconstructed = geo.Line3.from_cayley(*cayley)
print(f"Reconstructed direction: {reconstructed.direction}")

---
## 3. LineSegment3 — Bounded Segments

`LineSegment3` is a finite-length line segment defined by two endpoints.

**Properties:** `start_point`, `end_point`, `center_point`, `length`

**Methods:** `error()`, `flip()`, `endpoint_swap()`

In [ ]:
# --- Construct from endpoints ---
seg = geo.LineSegment3.from_endpoints(1, 2, 3, 4, 5, 6)

print(f"Start:  {seg.start_point()}")
print(f"End:    {seg.end_point()}")
print(f"Center: {seg.center_point()}")
print(f"Length: {seg.length:.4f}")

In [ ]:
# --- Segment operations ---
seg.flip()
print(f"Flipped direction: start={seg.start_point()}, end={seg.end_point()}")

seg.endpoint_swap()
print(f"Endpoints swapped: start={seg.start_point()}, end={seg.end_point()}")

# Fitting error (requires a ground-truth Line3)
gt_line = geo.Line3(1, 2, 3, 3, 3, 3)  # approximate line through the segment
print(f"Error vs ground-truth line: {seg.error(gt_line):.6f}")

---
## 4. Plane — Infinite Planes

A plane defined by a point on the plane and a normal vector.

**Construction:**
- `Plane(px, py, pz, nx, ny, nz)` — point + normal
- `Plane.from_three_points(p1, p2, p3)` — fit to three points

**Properties:** `normal`, `dist_to_origin`

**Methods:** `distance()`, `nearest_point()`, `intersects_line()`,
`intersection_with_line()`, `intersection_with_plane()`

In [ ]:
# --- Construct planes ---
# XY plane (z=0)
xy_plane = geo.Plane(0, 0, 0, 0, 0, 1)
print(f"XY plane normal: {xy_plane.normal}")
print(f"Distance to origin: {xy_plane.dist_to_origin:.4f}")

# Plane from three points (9 individual floats)
plane3 = geo.Plane.from_three_points(0, 0, 0, 1, 0, 0, 0, 1, 0)
print(f"From 3 points: normal={plane3.normal}")

In [ ]:
# --- Point-to-plane distance ---
dist = xy_plane.distance(5.0, 5.0, 7.0)
nearest = xy_plane.nearest_point(5.0, 5.0, 7.0)
print(f"Point (5, 5, 7) to XY plane:")
print(f"  Distance: {dist:.2f}")
print(f"  Nearest:  {nearest}")

---
## 5. Line-Plane Intersections

The `Plane` class can compute intersections with lines and other planes.

In [ ]:
# --- Line intersecting the XY plane ---
# Line from (0, 0, 5) pointing downward
line_down = geo.Line3(0, 0, 5, 0, 0, -1)

print(f"Line intersects XY plane? {xy_plane.intersects_line(line_down)}")
intersection = xy_plane.intersection_with_line(line_down)
print(f"Intersection point: {intersection}")

In [ ]:
# --- Plane-plane intersection ---
# XZ plane (y=0)
xz_plane = geo.Plane(0, 0, 0, 0, 1, 0)

# Intersection of XY and XZ planes = x-axis
inter_line = xy_plane.intersection_with_plane(xz_plane)
print(f"XY \u2229 XZ plane: direction={inter_line.direction}")
print(f"This is the x-axis: direction \u2248 [1, 0, 0]")

In [ ]:
# --- Parallel line (no intersection) ---
line_parallel = geo.Line3(0, 0, 5, 1, 0, 0)  # parallel to XY plane at z=5
print(f"Parallel line intersects XY plane? {xy_plane.intersects_line(line_parallel)}")

---
## 6. Pose — Position & Orientation

`Pose` combines translation (3 DOF) and rotation (3 DOF) into a rigid
body transform.

**Parameters:** `tx, ty, tz, rx, ry, rz` (rotation as Rodrigues angles)

**Properties:** `origin`, `orientation`

**Methods:** `rot_matrix()`, `hom_matrix()`, `translate()`, `rotate()`,
`rotate_around()`, `concat()`

In [ ]:
# --- Construct a pose ---
import math
pose = geo.Pose(1.0, 2.0, 3.0, 0.0, 0.0, math.radians(45))

print(f"Origin: {pose.origin}")
print(f"Orientation (Rodrigues): {pose.orientation}")
print(f"\nRotation matrix:\n{pose.rot_matrix()}")
print(f"\nHomogeneous matrix (4\u00d74):\n{pose.hom_matrix()}")

In [ ]:
# --- Transform operations ---
import math

pose2 = geo.Pose(0, 0, 0, 0, 0, math.radians(30))

# Concatenation is in-place
pose_combined = geo.Pose(1.0, 2.0, 3.0, 0.0, 0.0, math.radians(45))
pose_combined.concat(pose2)
print(f"Combined origin: {pose_combined.origin}")
print(f"Combined orientation: {pose_combined.orientation}")

# Translate a pose (in-place)
pose_moved = geo.Pose(1.0, 2.0, 3.0, 0.0, 0.0, math.radians(45))
pose_moved.translate(10, 0, 0)
print(f"\nTranslated origin: {pose_moved.origin}")

In [ ]:
# --- Create a camera ---
# Typical pinhole camera: 640x480, focal=500px
cam = geo.CameraCV(
    focal_x=500.0, focal_y=500.0,
    offset_x=320.0, offset_y=240.0,
    width=640, height=480,
    tx=0.0, ty=0.0, tz=5.0,    # 5 meters back
    rx=0.0, ry=0.0, rz=0.0     # looking down z-axis
)

print(f"Focal length: {cam.focal}")
print(f"Image offset: {cam.offset}")
print(f"Image size: {cam.image_size}")
print(f"Field of view: {np.degrees(cam.fov)} degrees")

---
## 7. Camera Models

The camera hierarchy builds on `Pose` to add intrinsic parameters:

```
Camera   (focal, offset, FOV, image_size)
  \u251c\u2500 CameraHom        \u2500 homogeneous point projection
  \u251c\u2500 CameraPluecker   \u2500 Pl\u00fccker line/segment projection
  \u251c\u2500 Camera2P         \u2500 two-point line/segment projection
  \u2514\u2500 CameraCV         \u2500 OpenCV-compatible + line projection
```

**Parameters:** `focal_x, focal_y, offset_x, offset_y, w, h, tx, ty, tz, rx, ry, rz`

In [ ]:
# --- Project a 3D point to 2D ---
point_2d = cam.project_point(1.0, 0.5, 8.0)
print(f"3D point: (1.0, 0.5, 8.0)")
print(f"2D projection: {point_2d}")

---
## 8. Projection — 3D to 2D

Different camera classes project different primitives:

| Camera Class | Projects |
|-------------|----------|
| `CameraHom` | Points (homogeneous) |
| `CameraPluecker` | Lines, segments (Pl\u00fccker) |
| `Camera2P` | Lines, segments (two-point) |
| `CameraCV` | Points + lines/segments (OpenCV) |

In [ ]:
# --- Project multiple 3D points ---
points_3d = [
    (0, 0, 8),
    (1, 0, 8),
    (0, 1, 8),
    (1, 1, 8),
    (0.5, 0.5, 6),
]

points_2d = cam.project_points(points_3d)
print("3D -> 2D projections:")
for p3, p2 in zip(points_3d, points_2d):
    print(f"  {p3} -> ({p2[0]:.1f}, {p2[1]:.1f})")

In [ ]:
# --- Project 3D line segments to 2D ---
seg3d_a = geo.LineSegment3.from_endpoints(0, 0, 8, 2, 0, 8)
seg3d_b = geo.LineSegment3.from_endpoints(0, 0, 8, 0, 2, 8)
seg3d_c = geo.LineSegment3.from_endpoints(0, 0, 6, 0, 0, 10)

# Project using CameraCV
segments_3d = [seg3d_a, seg3d_b, seg3d_c]
segments_2d = cam.project_line_segments(segments_3d)

print("Projected line segments:")
for i, seg2d in enumerate(segments_2d):
    ep = seg2d.end_points()
    print(f"  Seg {i}: ({ep[0]:.1f}, {ep[1]:.1f}) -> ({ep[2]:.1f}, {ep[3]:.1f})")

In [ ]:
# --- Visualize projected segments ---
fig, ax = plt.subplots(1, 1, figsize=(8, 6))
ax.set_xlim(0, 640)
ax.set_ylim(480, 0)  # image coordinates: y down
ax.set_aspect("equal")
ax.set_title("3D segments projected to 2D")
ax.set_xlabel("x (pixels)")
ax.set_ylabel("y (pixels)")

colors = ["red", "green", "blue"]
labels = ["Horizontal", "Vertical", "Depth"]
for seg2d, color, label in zip(segments_2d, colors, labels):
    ep = seg2d.end_points()
    ax.plot([ep[0], ep[2]], [ep[1], ep[3]], color=color, linewidth=2, label=label)

# Plot projected points
for p2 in points_2d:
    ax.plot(p2[0], p2[1], 'ko', markersize=5)

ax.legend()
plt.tight_layout()
plt.show()

---
## 9. Rerun 3D Visualization

If Rerun is installed, we can visualize lines, planes, and camera
poses in an interactive 3D viewer.

> **Note:** This section requires `rerun-sdk[notebook]`. If not installed,
> the cells will be skipped gracefully.

In [ ]:
if HAS_RERUN:
    rr.init("3D Geometry", spawn=False)
    rr.log("world", rr.ViewCoordinates.RIGHT_HAND_Z_UP, static=True)
    print("Rerun initialized")

In [ ]:
if HAS_RERUN:
    # --- Log coordinate axes ---
    rr.log("world/axes", rr.Arrows3D(
        origins=[[0, 0, 0], [0, 0, 0], [0, 0, 0]],
        vectors=[[3, 0, 0], [0, 3, 0], [0, 0, 3]],
        colors=[[255, 0, 0], [0, 255, 0], [0, 0, 255]],
        labels=["X", "Y", "Z"],
    ))

    # --- Log Line3 as a long segment ---
    p = line_a.origin
    d = line_a.direction
    rr.log("world/line_a", rr.LineStrips3D(
        [[[p[0] - 5*d[0], p[1] - 5*d[1], p[2] - 5*d[2]],
          [p[0] + 5*d[0], p[1] + 5*d[1], p[2] + 5*d[2]]]],
        colors=[[255, 165, 0]],
        labels=["Line A (x-axis)"],
    ))

    p = line_b.origin
    d = line_b.direction
    rr.log("world/line_b", rr.LineStrips3D(
        [[[p[0] - 5*d[0], p[1] - 5*d[1], p[2] - 5*d[2]],
          [p[0] + 5*d[0], p[1] + 5*d[1], p[2] + 5*d[2]]]],
        colors=[[0, 200, 200]],
        labels=["Line B (diagonal)"],
    ))

    print("Logged Line3 objects")

In [ ]:
if HAS_RERUN:
    # --- Log LineSegment3 objects ---
    colors_str = ["red", "green", "blue"]
    for i, s3d in enumerate(segments_3d):
        sp = s3d.start_point()
        ep = s3d.end_point()
        rr.log(f"world/segment_{i}", rr.LineStrips3D(
            [[[sp[0], sp[1], sp[2]], [ep[0], ep[1], ep[2]]]],
            colors=[colors_str[i]],
            labels=[labels[i]],
            radii=[0.03],
        ))

    # --- Log points ---
    pts_array = np.array(points_3d, dtype=np.float64)
    rr.log("world/points", rr.Points3D(
        pts_array,
        colors=[[200, 200, 200]] * len(points_3d),
        radii=[0.05],
    ))

    print("Logged segments and points")

In [ ]:
if HAS_RERUN:
    # --- Log the XY plane as a semi-transparent quad ---
    plane_corners = np.array([
        [-4, -4, 0], [4, -4, 0], [4, 4, 0], [-4, 4, 0]
    ], dtype=np.float32)
    rr.log("world/xy_plane", rr.Mesh3D(
        vertex_positions=plane_corners,
        triangle_indices=[[0, 1, 2], [0, 2, 3]],
        vertex_colors=[[100, 100, 255, 80]] * 4,
    ))

    # --- Log intersection point ---
    rr.log("world/intersection", rr.Points3D(
        [intersection],
        colors=[[255, 255, 0]],
        radii=[0.08],
        labels=["Line-Plane intersection"],
    ))

    print("Logged plane and intersection")

In [ ]:
if HAS_RERUN:
    # --- Log camera frustum ---
    rr.log("world/camera", rr.Pinhole(
        focal_length=[500.0, 500.0],
        principal_point=[320.0, 240.0],
        resolution=[640, 480],
    ))
    # Camera pose (translation and rotation)
    rr.log("world/camera", rr.Transform3D(
        translation=[0.0, 0.0, 5.0],
    ))

    print("Logged camera")

In [ ]:
if HAS_RERUN:
    rr.notebook_show(width=1200, height=800)
else:
    print("Rerun not available \u2014 skipping 3D viewer")

---
## 10. Comprehension Questions

1. How does `Line3` differ from `LineSegment3`? When would you use each?
2. What does the `momentum` property of `Line3` represent geometrically?
3. Why can two non-parallel planes always intersect in a line?
4. What is the Cayley representation and why is it useful?
5. What is the difference between `CameraPluecker` and `Camera2P` projections?

---
## 11. Challenge Exercise

**Task:** Build a 3D wireframe cube and project it.

1. Define the 8 corners of a unit cube centered at (0, 0, 8)
2. Create 12 `LineSegment3` objects for the cube edges
3. Set up a `CameraCV` looking at the cube
4. Project all edges to 2D and plot them
5. *(Bonus)* Log everything to Rerun for interactive viewing

In [ ]:
# Your code here:
# ...


---
## Summary

| Component | Description | Key Methods |
|-----------|-------------|-------------|
| `Line3` | Infinite 3D line | `distance_to_point()`, `angle_to()`, `cayley()` |
| `LineSegment3` | Bounded segment | `start_point`, `end_point`, `length` |
| `Plane` | Infinite plane | `distance()`, `intersects_line()`, `intersection_with_*()` |
| `Pose` | 6-DOF rigid transform | `rot_matrix()`, `hom_matrix()`, `concat()` |
| `Camera` | Intrinsic + extrinsic | `focal`, `fov`, `image_size` |
| `CameraCV` | OpenCV-compatible | `project_point()`, `project_segments()` |
| `CameraPluecker` | Pl\u00fccker projection | `project_line()`, `project_segment()` |
| `Camera2P` | Two-point projection | `project_line()`, `project_segment()` |

### Next Steps

- **Tutorial 5**: Advanced gradient & quadrature filters
- **Tutorial 6**: Image operators & pipelines
- Use camera models to project detected line segments back into 3D
- Explore the Rerun viewer for debugging 3D reconstruction pipelines